# WSI Benchmark
This notebook compares the communities of the 30 words for each algorithm against the Gold Standard communities.

## Import

In [1]:
!pip install --upgrade pip
!pip install "pymongo[srv]"

  Attempting uninstall: pip
    Found existing installation: pip 21.2.4
    Uninstalling pip-21.2.4:
      Successfully uninstalled pip-21.2.4


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\ASUS\\AppData\\Local\\Temp\\pip-uninstall-84euib1e\\pip.exe'
Consider using the `--user` option or check the permissions.



     -------------------------------------- 365.3/365.3 kB 1.7 MB/s eta 0:00:00
     -------------------------------------- 269.1/269.1 kB 2.8 MB/s eta 0:00:00


In [4]:
import json
from bson import json_util
import os
import sys
import pandas as pd
import pymongo
from pymongo import MongoClient
from bson.objectid import ObjectId
from itertools import combinations, chain
from math import factorial, log
import numpy as np
from sklearn.metrics.cluster import normalized_mutual_info_score as nmi
from sklearn.metrics import f1_score
from sklearn.metrics import rand_score, adjusted_rand_score
from sklearn.metrics import accuracy_score
from sklearn.metrics.cluster import contingency_matrix

In [5]:
temp_path = os.getcwd().split('\\')
project_dir = '\\'.join(temp_path[:temp_path.index('SOURCE') + 1])

In [6]:
sys.path.insert(0, f'{project_dir}/code/Packages')

In [7]:
def purity_score(y_true, y_pred):
    # compute contingency matrix (also called confusion matrix)
    contingency_matrix_res = contingency_matrix(y_true, y_pred)
    # return purity
    return np.sum(np.amax(contingency_matrix_res, axis=0)) / np.sum(contingency_matrix_res) 

## Load communities 

Load nodelist

In [10]:
directory = f'{project_dir}/data/3 - Network Generation/'

In [11]:
# Open node list
nodelist = pd.read_pickle(directory+'final_edgelists/nodelist.pkl')
res = dict((v,k) for k,v in nodelist.items())
# {ego:1, community: [[1,2],[3,4]}

Load communities and transform them to word form

In [12]:
def json_to_dict(filename):
  with open(filename) as json_file:
    data = json.load(json_file)
    data['ego'] = res.get(data['ego'])
    # print("Ego:", data['ego'])
    # Convert to list of words
    wordlist = data['community'].items()
    commlist = []
    for key, community in wordlist:
      new_dict = {}
      new_dict['id'] = key
      new_dict['context_words'] = [res.get(x) for x in community]
      commlist.append(new_dict)
    data['community'] = commlist
    # print("Communities:", data['community'])
    return data
def assign_algorithm(filename):
  algo = filename.split('_')
  if algo[1] == "leiden":
    return algo[1] + "_" + algo[2]
  else:
    return algo[1]

In [ ]:
comm_dir = f'{project_dir}/data/5 - Validation/'

In [ ]:
wsi_comms = [] # Stores all the communities in sentence format instead of IDs
for file in os.listdir(comm_dir+"communities/30 words/filtered_10/"):
     filename = os.fsdecode(file)
     if filename.endswith(".json"): 
       data = json_to_dict(comm_dir+"communities/30 words/filtered_10/"+filename)
       data['algorithm'] = assign_algorithm(filename)
       wsi_comms.append(data)

## Load DB sentences

In [ ]:
# Kindly request access from the FilWordNet Team to get your PUBLIC_URL
import requests 

PUBLIC_URL = ""
API_CALL = "/get_gs_sentences"

res = requests.get(PUBLIC_URL+API_CALL)
word_list = list(res.json())

In [ ]:
# client = pymongo.MongoClient("")
# db = client[""]
# word_collection = db[""]
# word_list = list(word_collection.find())

In [ ]:
cohfie_sentences = pd.read_excel(f"{directory}example_sentences_50.xlsx", sheet_name="final sentences")
# cohfie_sentences = pd.read_csv(f"{directory}gold_standard/example_sentences_50.csv")
cohfie_sentences = cohfie_sentences.rename(columns={'original sentence from COHFIE': 'original_sentence'})

In [ ]:
cohfie_sentences = cohfie_sentences[['word', 'source', 'sent_id', 'original_sentence']]

In [ ]:
cohfie_sentences

,word,source,sent_id,original_sentence
0,pinto,news_sites,news_6731,Nagpapasalamat ako sa Dios sa biyaya na biniga...
1,pinto,online_forums,online_forums_318787,Yung pinto namin nala-lock na ng maayos .
2,pinto,social_media,social_media_94436,Ikaw parin ang susi sa pinto ng iyong tadhana
3,pinto,online_forums,online_forums_431768,I plan on going to Pinto Art tomorrow but I co...
4,pinto,social_media,social_media_9703458,"Akalain mo yun pag pasok ko sa pinto , nasa la..."
...,...,...,...,...
638,sinag,news_sites,news_359333,Parang wala nang katapusan ang mga paghamon sa...
639,sinag,online_forums,online_forums_351963,Depende sa sinag ng araw yung 'cute' pero wala...
640,sinag,online_forums,online_forums_102685,"Naririnig mo ang huni ng ibon , ang mga tunog ..."
641,sinag,social_media,social_media_7556759,bawat sulok ata ng bahay namin may sinag ng ar...


## WSD DB Sentences to WSI Comms (don't have to run)

In [ ]:
import wsd_nopos as wsd
import unicodedata
import regex as re

# Code in this cell is from the package
def jaccard_similarity(comm1, comm2):
    intersection = len(set(comm1).intersection(set(comm2)))
    #union = len(set(comm1)) + len(set(comm2)) - intersection
    union = len(set(comm1).union(set(comm2)))
    
    # if union == 0 or float(intersection) / union < 0.1:
    #   return -1
    # else:
    #   return float(intersection) / union

    return float(intersection) / union if union > 0 else -1
    #return len(comm1) * intersection / union if union > 0 else -1

def get_similar_comm(context_words, comms, target):
    similarity_scores = []
    for i in range(len(comms)):
      sim = jaccard_similarity(word_tokenize_with_filter(context_words, target), comms[i]['context_words'])
      similarity_scores.append(sim)
      # print("community id: ",i, sim)
      max_score = max(similarity_scores)
      if max_score != -1:
        max_index = similarity_scores.index(max_score)
      else:
        max_index = -1
    #print(f"Max is {max_score} of Comm ID {max_index}")
    return max_index

def word_sense_disambiguation(sentences, comms):
    final_scores = {}
    scores = {}
    for sent in sentences:
      label = get_similar_comm(sent, comms['community'], comms['ego'])
      scores[sent] = label
    return scores, comms['algorithm'], comms['ego']


def word_tokenize_with_filter(sentence, target):
    words = []
    sentence = sentence.split(' ')
    sent = [wsd.normalize(word.lower().strip()).strip() for word in sentence]
    for index in range(len(sent)):
        word = sent[index]
        if wsd.is_valid(word):
            words.append(word)
    # Manipulate indices to just retain the window size 3 and REMOVE the target word
    return words

In [ ]:
wsd.download_stopwords()

Finished downloading stopwords_tl.
Finished downloading stopwords_en.


In [ ]:
# create dictionary of sentence ids and sentences
sentid_dict = dict(list(zip(cohfie_sentences['original_sentence'], cohfie_sentences['sent_id'])))

In [ ]:
target_words_30 = ['pinto', 'sikat', 'dilim', 'sinag', 'kulay', 'sarap', 'tali', 'bayad', 'buto', 'suka', 'sumpa', 'patay', 'kupas', 'sipa', 'lason', 'salita', 'santo', 'lunod', 'alam', 'parusa', 'dahon', 'bato', 'dahan', 'ginhawa', 'susi', 'sulat', 'kulong', 'kalat', 'buwan', 'hamon']
# target_words_30 = ['sipa'] 

In [ ]:
for word in target_words_30:
  #test_db_sentences = word_list['label' == 'pinto']
  test_db_sentences = list(cohfie_sentences[cohfie_sentences['word'] == word]['original_sentence'])
  comms = list(filter(lambda comm: comm['ego'] == word, wsi_comms))

  word_results = []
  for comm in comms:
    #res,algo, ego = word_sense_disambiguation(test_db_sentences['sentences'], comm)
    res,algo, ego = word_sense_disambiguation(test_db_sentences, comm)
    temp = {}
    temp['word'] = ego
    temp['algorithm'] = algo
    temp['results'] = [(k,v) for k,v in res.items()]
    word_results.append(temp)

  #print(word_results)
  wsd_res_df = pd.DataFrame(word_results).explode('results')
  wsd_res_df['sentence'], wsd_res_df['label'] = zip(*wsd_res_df.results)
  wsd_res_df = wsd_res_df.drop('results',axis=1)
  wsd_res_df.reset_index(drop=True)

  wsd_res_df['sent_id'] = wsd_res_df['sentence'].map(sentid_dict)
  wsd_res_df = wsd_res_df[['word', 'algorithm', 'sent_id', 'sentence', 'label']]
  wsd_res_df['key'] = wsd_res_df['algorithm'] + "_" + wsd_res_df['label'].apply(str)


  # display(wsd_res_df)
  wsd_res_dict = wsd_res_df.groupby(['key'])['sent_id'].apply(list).to_dict()
  # display(wsd_res_dict)
  
  with open(f'{comm_dir}/Gold Standard/{word}.json', 'w') as fp:
        json.dump(wsd_res_dict, fp)


  ### For SKLEARN formatting
  wsd_res_df.to_csv(f'{directory}benchmark/{word}.csv',index=False)

In [ ]:
wsd_res_df

,word,algorithm,sent_id,sentence,label,key
0,hamon,leiden_cpm,social_media_10116142,Magpatuloy sa lahat ng hamon . Kaya mo yan ......,0,leiden_cpm_0
0,hamon,leiden_cpm,social_media_1290137,"Tinde ng mga hamon ng buhay , pero sige palag .",0,leiden_cpm_0
0,hamon,leiden_cpm,social_media_7709904,"Celso Suquib , Representative , Integrated Fis...",0,leiden_cpm_0
0,hamon,leiden_cpm,news_1598308,"Napaka-grounded niya , marunong makisama at ga...",0,leiden_cpm_0
0,hamon,leiden_cpm,books_107265,Ang lantarang pagpapagalit ng tao sa Diyos ay ...,0,leiden_cpm_0
...,...,...,...,...,...,...
3,hamon,louvain,online_forums_179396,-Jay Taruc-Howie Severino-Sandra Aguinaldo-Kar...,4,louvain_4
3,hamon,louvain,online_forums_67179,"I say things like : ""Gusto ko hamon tostado an...",0,louvain_0
3,hamon,louvain,news_784270,"I'm challenging Archbishop Cruz , knowing his ...",0,louvain_0
3,hamon,louvain,news_1130517,Matapang na tinanggap ni Nadine Lustre ang ham...,3,louvain_3


In [ ]:
wsd_res_df[wsd_res_df['label'] == -1]

,word,algorithm,sent_id,sentence,label,key


## Load Gold Standard

In [ ]:
target_words_30 = ['pinto', 'sikat', 'dilim', 'sinag', 'kulay', 'sarap', 'tali', 'bayad', 'buto', 'suka', 'sumpa', 'patay', 'kupas', 'sipa', 'lason', 'salita', 'santo', 'lunod', 'alam', 'parusa', 'dahon', 'bato', 'dahan', 'ginhawa', 'susi', 'sulat', 'kulong', 'kalat', 'buwan', 'hamon']
#target_words_30 = ['sipa'] 

In [ ]:
# Load gold standard
gs_dict = {}
for word in target_words_30:
  with open(f'{comm_dir}/Gold Standard/{word}.json') as json_file:
    gold_standard = json.load(json_file)
    # gold_standard_df = pd.json_normalize(gold_standard).transpose().rename(columns={0: 'sent_id'})
    # display(gold_standard_df)

    gs_dict[word] = list(gold_standard.values())

## Comparing with Gold Standard

Load WSD results

In [ ]:
target_words_30 = ['pinto', 'sikat', 'dilim', 'sinag', 'kulay', 'sarap', 'tali', 'bayad', 'buto', 'suka', 'sumpa', 'patay', 'kupas', 'sipa', 'lason', 'salita', 'santo', 'lunod', 'alam', 'parusa', 'dahon', 'bato', 'dahan', 'ginhawa', 'susi', 'sulat', 'kulong', 'kalat', 'buwan', 'hamon']
# target_words_30 = ['sipa'] 

In [ ]:
def add_comm_label(sent_id, comms):
  for index, comm in enumerate(comms):
    if sent_id in comm:
      return index

In [ ]:
def purity_score(y_true, y_pred):
    # compute contingency matrix (also called confusion matrix)
   # print(y_true)
    #print(y_pred)
    contingency_matrix_res = contingency_matrix(y_true, y_pred)
    # return purity
    return np.sum(np.amax(contingency_matrix_res, axis=0)) / np.sum(contingency_matrix_res) 

In [ ]:
print("Average Number of Clusters:\n")
print(f"CW: {cw_total_clusters / 30}")
print(f"Leiden Mod: {leidenmod_total_clusters / 30}")
print(f"Leiden CPM: {leidencpm_total_clusters / 30}")
print(f"Louvain: {louvain_total_clusters / 30}")

sum = 0
for val in gs_dict.values():
  sum += len(val)

print(f"GS: {sum / 30}")

Average Number of Clusters:

CW: 1.0
Leiden Mod: 3.6
Leiden CPM: 1.0
Louvain: 3.4
GS: 3.066666666666667


In [ ]:
total_df = pd.DataFrame()

cw_total_clusters = 0
leidenmod_total_clusters = 0
leidencpm_total_clusters = 0
louvain_total_clusters = 0

for word in target_words_30:
  #print(word)
  # Load WSD results
  with open(f'{comm_dir}/Gold Standard/{word}.json') as json_file:
    wsd_res = json.load(json_file)
    wsd_res_df = pd.json_normalize(wsd_res).transpose().reset_index().rename(columns={'index': 'key', 0: 'sent_ids'})

  wsd_res_df['label'] = wsd_res_df['key'].apply(lambda x: x.split('_', 2)[-1])
  wsd_res_df['algorithm'] = wsd_res_df.apply(lambda x: x.key.replace("_" + x.label, ''), axis=1)
  wsd_res_df = wsd_res_df[['key', 'algorithm', 'label', 'sent_ids']]
  #display(wsd_res_df)

  cw_total_clusters += wsd_res_df[wsd_res_df['algorithm'] == 'cw'].shape[0]
  leidenmod_total_clusters += wsd_res_df[wsd_res_df['algorithm'] == 'leiden_modularity'].shape[0]
  leidencpm_total_clusters += wsd_res_df[wsd_res_df['algorithm'] == 'leiden_cpm'].shape[0]
  louvain_total_clusters += wsd_res_df[wsd_res_df['algorithm'] == 'louvain'].shape[0]

  # get list of communities for each algorithm
  cw_list = list(wsd_res_df[wsd_res_df['algorithm'] == 'cw']['sent_ids'])
  leiden_mod_list = list(wsd_res_df[wsd_res_df['algorithm'] == 'leiden_modularity']['sent_ids'])
  leiden_cpm_list = list(wsd_res_df[wsd_res_df['algorithm'] == 'leiden_cpm']['sent_ids'])
  louvain_list = list(wsd_res_df[wsd_res_df['algorithm'] == 'louvain']['sent_ids'])

  # create dataframe for each algorithm
  cw_df = pd.DataFrame(columns=['sent_id', 'wsd_label'])
  leiden_mod_df = pd.DataFrame(columns=['sent_id', 'wsd_label'])
  leiden_cpm_df = pd.DataFrame(columns=['sent_id', 'wsd_label'])
  louvain_df= pd.DataFrame(columns=['sent_id', 'wsd_label'])

  # get sentence id list
  sent_id_list = list(chain(*gs_dict[word]))

  cw_df['sent_id'] = sent_id_list
  leiden_mod_df['sent_id'] = sent_id_list
  leiden_cpm_df['sent_id'] = sent_id_list
  louvain_df['sent_id'] = sent_id_list

  # add label to gold standard df
  gs_df = pd.DataFrame(columns=['sent_id', 'gs_label'])
  gs_df['sent_id'] = sent_id_list
  gs_df['gs_label'] = gs_df['sent_id'].apply(add_comm_label, args=(gs_dict[word], ))

  # add community label for each sentence for each algorithm
  cw_df['wsd_label'] = cw_df['sent_id'].apply(add_comm_label, args=(cw_list, ))
  leiden_mod_df['wsd_label'] = leiden_mod_df['sent_id'].apply(add_comm_label, args=(leiden_mod_list, ))
  leiden_cpm_df['wsd_label'] = leiden_cpm_df['sent_id'].apply(add_comm_label, args=(leiden_cpm_list, ))
  louvain_df['wsd_label'] = louvain_df['sent_id'].apply(add_comm_label, args=(louvain_list, ))

  # join gs_df with the algorithm dfs
  cw_df = pd.merge(cw_df, gs_df, on='sent_id')
  leiden_mod_df = pd.merge(leiden_mod_df, gs_df, on='sent_id')
  leiden_cpm_df = pd.merge(leiden_cpm_df, gs_df, on='sent_id')
  louvain_df = pd.merge(louvain_df, gs_df, on='sent_id')

  # result df
  #comparison_res_df = pd.DataFrame({'algorithm': ['cw', 'leiden_mod', 'leiden_cpm', 'louvain'], 'purity': None, 'f_measure': None, 'nmi': None, 'rand_index': None, 'accuracy':None})
  #comparison_res_df = pd.DataFrame({'algorithm': ['cw', 'leiden_mod', 'leiden_cpm', 'louvain'], 'purity': None, 'f_measure': None, 'nmi': None, 'rand_index': None, 'adjusted_rand_index': None, 'accuracy':None})
  comparison_res_df = pd.DataFrame({'algorithm': ['cw', 'leiden_mod', 'leiden_cpm', 'louvain'], 'purity': None, 'inverse_purity': None, 'f_measure': None, 'nmi': None, 'rand_index': None})

  # perform metrics
  # cw
  comparison_res_df.iloc[0]['purity'] = purity_score(list(cw_df['gs_label']), list(cw_df['wsd_label']))
  comparison_res_df.iloc[0]['inverse_purity'] = purity_score(list(cw_df['wsd_label']), list(cw_df['gs_label']))
  #comparison_res_df.iloc[0]['f_measure'] = f1_score(list(cw_df['gs_label']), list(cw_df['wsd_label']), average='weighted')
  comparison_res_df.iloc[0]['f_measure'] = 2 * comparison_res_df.iloc[0]['purity'] * comparison_res_df.iloc[0]['inverse_purity'] / (comparison_res_df.iloc[0]['purity'] + comparison_res_df.iloc[0]['inverse_purity'])
  comparison_res_df.iloc[0]['nmi'] = nmi(list(cw_df['gs_label']), list(cw_df['wsd_label']))
  comparison_res_df.iloc[0]['rand_index'] = rand_score(list(cw_df['gs_label']), list(cw_df['wsd_label']))
  #comparison_res_df.iloc[0]['adjusted_rand_index'] = adjusted_rand_score(list(cw_df['gs_label']), list(cw_df['wsd_label']))
  #comparison_res_df.iloc[0]['accuracy'] = accuracy_score(list(cw_df['gs_label']), list(cw_df['wsd_label']))

  # leiden mod
  comparison_res_df.iloc[1]['purity'] = purity_score(list(leiden_mod_df['gs_label']), list(leiden_mod_df['wsd_label']))
  comparison_res_df.iloc[1]['inverse_purity'] = purity_score(list(leiden_mod_df['wsd_label']), list(leiden_mod_df['gs_label']))
  #comparison_res_df.iloc[1]['f_measure'] = f1_score(list(leiden_mod_df['gs_label']), list(leiden_mod_df['wsd_label']), average='weighted')
  comparison_res_df.iloc[1]['f_measure'] = 2 * comparison_res_df.iloc[1]['purity'] * comparison_res_df.iloc[1]['inverse_purity'] / (comparison_res_df.iloc[1]['purity'] + comparison_res_df.iloc[1]['inverse_purity'])
  comparison_res_df.iloc[1]['nmi'] = nmi(list(leiden_mod_df['gs_label']), list(leiden_mod_df['wsd_label']))
  comparison_res_df.iloc[1]['rand_index'] = rand_score(list(leiden_mod_df['gs_label']), list(leiden_mod_df['wsd_label']))
  #comparison_res_df.iloc[1]['adjusted_rand_index'] = adjusted_rand_score(list(leiden_mod_df['gs_label']), list(leiden_mod_df['wsd_label']))
  #comparison_res_df.iloc[1]['accuracy'] = accuracy_score(list(leiden_mod_df['gs_label']), list(leiden_mod_df['wsd_label']))

  # leiden cpm
  comparison_res_df.iloc[2]['purity'] = purity_score(list(leiden_cpm_df['gs_label']), list(leiden_cpm_df['wsd_label']))
  comparison_res_df.iloc[2]['inverse_purity'] = purity_score(list(leiden_cpm_df['wsd_label']), list(leiden_cpm_df['gs_label']))
  #comparison_res_df.iloc[2]['f_measure'] = f1_score(list(leiden_cpm_df['gs_label']), list(leiden_cpm_df['wsd_label']), average='weighted')
  comparison_res_df.iloc[2]['f_measure'] = 2 * comparison_res_df.iloc[2]['purity'] * comparison_res_df.iloc[2]['inverse_purity'] / (comparison_res_df.iloc[2]['purity'] + comparison_res_df.iloc[2]['inverse_purity'])
  comparison_res_df.iloc[2]['nmi'] = nmi(list(leiden_cpm_df['gs_label']), list(leiden_cpm_df['wsd_label']))
  comparison_res_df.iloc[2]['rand_index'] = rand_score(list(leiden_cpm_df['gs_label']), list(leiden_cpm_df['wsd_label']))
  #comparison_res_df.iloc[2]['adjusted_rand_index'] = adjusted_rand_score(list(leiden_cpm_df['gs_label']), list(leiden_cpm_df['wsd_label']))
  #comparison_res_df.iloc[2]['accuracy'] = accuracy_score(list(leiden_cpm_df['gs_label']), list(leiden_cpm_df['wsd_label']))

  # louvain
  comparison_res_df.iloc[3]['purity'] = purity_score(list(louvain_df['gs_label']), list(louvain_df['wsd_label']))
  comparison_res_df.iloc[3]['inverse_purity'] = purity_score(list(louvain_df['wsd_label']), list(louvain_df['gs_label']))
  #comparison_res_df.iloc[3]['f_measure'] = f1_score(list(louvain_df['gs_label']), list(louvain_df['wsd_label']), average='weighted')
  comparison_res_df.iloc[3]['f_measure'] = 2 * comparison_res_df.iloc[3]['purity'] * comparison_res_df.iloc[3]['inverse_purity'] / (comparison_res_df.iloc[3]['purity'] + comparison_res_df.iloc[3]['inverse_purity'])
  comparison_res_df.iloc[3]['nmi'] = nmi(list(louvain_df['gs_label']), list(louvain_df['wsd_label']))
  comparison_res_df.iloc[3]['rand_index'] = rand_score(list(louvain_df['gs_label']), list(louvain_df['wsd_label']))
  #comparison_res_df.iloc[3]['adjusted_rand_index'] = adjusted_rand_score(list(louvain_df['gs_label']), list(louvain_df['wsd_label']))
  #comparison_res_df.iloc[3]['accuracy'] = accuracy_score(list(louvain_df['gs_label']), list(louvain_df['wsd_label']))


  # TODO: save results somewhere or collate all comparison_res_dfs for all words
  #print("Word: ", word)
  #display(comparison_res_df)

  total_df = pd.concat([total_df, comparison_res_df])

In [ ]:
total_df

,algorithm,purity,inverse_purity,f_measure,nmi,rand_index
0,cw,0.72,1.0,0.837209,0.0,0.56
1,leiden_mod,0.72,0.44,0.546207,0.101643,0.483333
2,leiden_cpm,0.72,1.0,0.837209,0.0,0.56
3,louvain,0.72,0.52,0.603871,0.118649,0.506667
0,cw,0.727273,1.0,0.842105,0.0,0.584416
...,...,...,...,...,...,...
3,louvain,0.818182,0.409091,0.545455,0.170379,0.428571
0,cw,0.695652,1.0,0.820513,0.0,0.509881
1,leiden_mod,0.73913,0.347826,0.473043,0.140138,0.498024
2,leiden_cpm,0.695652,1.0,0.820513,0.0,0.509881


In [ ]:
total_df.groupby('algorithm').mean()

,purity,inverse_purity,f_measure,nmi,rand_index
algorithm,,,,,
cw,0.606433,1.000000,0.746186,0.033333,0.469477
leiden_cpm,0.606433,1.000000,0.746186,0.033333,0.469477
leiden_mod,0.692742,0.586797,0.622423,0.220567,0.545699
louvain,0.689058,0.599358,0.627663,0.217874,0.543061


## Playground

In [ ]:
truth = [1, 1, 1, 2, 3, 3]
pred = [2, 2, 2, 3, 1, 1]

print(f"purity: {purity_score(truth, pred)}")
print(f"fmeasure: {2 * purity_score(truth, pred) * purity_score(pred, truth) / (purity_score(truth, pred) + purity_score(pred, truth))}")

purity: 1.0
fmeasure: 1.0


In [ ]:
word = "sipa"

In [ ]:
word_df = pd.read_csv(f'{directory}benchmark/{word}.csv')
louvain_df = word_df[word_df['algorithm'] == 'louvain'].drop(columns=['word','key']).rename(columns={'label':'pred'})

# get sentence id list
sent_id_list = list(chain(*gs_dict[word]))

# create gold standard df with index as label
gs_df = pd.DataFrame(columns=['sent_id', 'gs_label'])
gs_df['sent_id'] = sent_id_list
gs_df['gs_label'] = gs_df['sent_id'].apply(add_comm_label, args=(gs_dict[word], ))

louvain_df = pd.merge(louvain_df,gs_df ,on=["sent_id"])
louvain_df = louvain_df.sort_values(['gs_label', 'pred'], ascending=[True, True]).rename(columns={'gs_label':'truth'})
# display(louvain_df.head(5))
y_pred = list(louvain_df['pred'])
y_truth = list(louvain_df['truth'])

print("Purity: ", purity_score(y_truth, y_pred))
print("Accuracy: ", accuracy_score(y_truth,y_pred))
print("F1: ", f1_score(y_truth, y_pred, average='weighted'))
print("NMI: ",  nmi(y_truth, y_pred))
print("RI: ", rand_score(y_truth, y_pred))